# OCR IAM-line — bản thực hành cơ bản

Nhận dạng từng dòng chữ viết tay tiếng Anh; primary CER ↓, bổ sung WER ↓. Hai biến thể cùng split, seed và chuẩn hóa. CNN + vanilla RNN + CTC học từ đầu, là điểm bắt đầu thực hành đầy đủ.

Chạy độc lập: **Restart Kernel → Run All**. Các trọng số ngôn ngữ/tokenizer đều học từ train. Không đọc notebook khác, fewshot hay workbook. Chưa huấn luyện đầy đủ; không kèm điểm benchmark đã đo.

**Bản basic cập nhật:** Ảnh grayscale 0..1, bỏ inversion; giữ resize theo tỷ lệ và padding trắng để CTC có đủ bước và đúng length. Không phải bản tối ưu; không cam kết chênh lệch điểm.

**Cách viết training/checkpoint:** training loop được viết trực tiếp trong cell (sklearn dùng fit); lưu gọn khi validation cải thiện. Checkpoint phục vụ inference, không lưu optimizer/scheduler/RNG để resume.


## 1. Chuẩn bị thư viện

Dùng PyTorch/torchvision tương thích của kernel. Thiếu datasets sẽ được cài. Không thay driver hay tự thay wheel CUDA.


In [ ]:
import importlib.util
import subprocess
import sys
NEEDED_PACKAGES = [('datasets', 'datasets==4.8.5')]
missing = [package for module, package in NEEDED_PACKAGES if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
if importlib.util.find_spec("torch") is None or importlib.util.find_spec("torchvision") is None:
    raise RuntimeError("Cần kernel có PyTorch và torchvision tương thích; cài bản CPU/CUDA phù hợp rồi chạy lại.")

from pathlib import Path
from collections import Counter, defaultdict
import csv
import hashlib
import io
import json
import math
import random
import re
import unicodedata
import numpy as np
from PIL import Image, ImageOps
import torch
import torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm


## 2. Cấu hình cố định

Thiết kế cho một GPU CUDA 32 GB, dùng độ chính xác PyTorch mặc định. Có thể sửa hằng số thủ công trước Run All. Bản tối ưu tải ResNet18 ImageNet V1 chỉ làm encoder ảnh; không tải OCR/tokenizer pretrained.


In [ ]:
SEED = 42
VARIANT = "basic"
ARTIFACT_DIR = Path("artifacts/ocr_iam_line") / VARIANT
CACHE_DIR = ARTIFACT_DIR / "data_cache"
WEIGHTS_CACHE_DIR = ARTIFACT_DIR / "weights"
CHECKPOINT_PATH = ARTIFACT_DIR / "best_model.pt"
PRETRAINING_POLICY = "scratch"
PRETRAINED_COMPONENTS = []
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
torch.hub.set_dir(str(WEIGHTS_CACHE_DIR))  # Chỉ đặt nơi cache, không dùng torch.hub.load.
print(f"Thiết bị: {DEVICE}; biến thể: {VARIANT}; PyTorch {torch.__version__}; torchvision {torchvision.__version__}")


SOURCE = {'row': 1, 'task': 'OCR dòng chữ viết tay tiếng Anh', 'dataset': 'Teklia/IAM-line', 'revision': 'fbdad97500ce54635c0d1ba306bf535cb40656cf', 'url': 'https://huggingface.co/datasets/Teklia/IAM-line'}
DATASET_ID = SOURCE["dataset"]
DATASET_REVISION = SOURCE["revision"]
USE_LOCAL_DATA = False
LOCAL_DATA_DIR = Path("data/iam_line_saved")
IMAGE_HEIGHT = 64
BATCH_SIZE = 16
EVAL_BATCH_SIZE = BATCH_SIZE
HIDDEN_DIM = 128
DROPOUT = 0.0
EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 0.001
BACKBONE_LEARNING_RATE = 0.0001
WEIGHT_DECAY = 0.0001
GRAD_CLIP = 5.0
PRIMARY_METRIC = "cer"
DIRECTION = "min"


In [ ]:
def fingerprint(records):
    payload = json.dumps(records, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode()).hexdigest()

def open_rgb(value):
    if isinstance(value, Image.Image):
        return ImageOps.exif_transpose(value).convert("RGB")
    if isinstance(value, dict):
        if value.get("bytes") is not None:
            return ImageOps.exif_transpose(Image.open(io.BytesIO(value["bytes"]))).convert("RGB")
        value = value.get("path")
    if value is None:
        raise ValueError("Thiếu ảnh; cần PIL.Image, bytes ảnh hoặc đường dẫn thật.")
    with Image.open(value) as image:
        return ImageOps.exif_transpose(image).convert("RGB")

def to_device(batch):
    return {key: value.to(DEVICE) if torch.is_tensor(value) and key not in ("lengths", "widths", "target_lengths") else value
            for key, value in batch.items()}

def make_loader(dataset, train=False):
    return DataLoader(dataset, batch_size=BATCH_SIZE if train else EVAL_BATCH_SIZE,
                      shuffle=train, num_workers=0, collate_fn=collate_batch,
                      generator=torch.Generator().manual_seed(SEED))


## 3. Dữ liệu và split

Giữ nguyên train/validation/test của Teklia/IAM-line, đúng revision. Dòng chữ viết tay tiếng Anh; không đại diện chữ in hoặc OCR tiếng Việt. Không suy writer_id từ thứ tự hàng.


In [ ]:
from datasets import load_dataset, load_from_disk, Image as HFImage
try:
    raw = load_from_disk(str(LOCAL_DATA_DIR)) if USE_LOCAL_DATA else load_dataset(
        DATASET_ID, revision=DATASET_REVISION, cache_dir=str(CACHE_DIR))
except Exception as exc:
    raise RuntimeError("Không tải được Teklia/IAM-line. Kiểm tra mạng/quyền truy cập, hoặc cung cấp "
                       "DatasetDict save_to_disk đúng nguồn tại LOCAL_DATA_DIR và bật USE_LOCAL_DATA. "
                       f"Không có dữ liệu thay thế tự động. Chi tiết: {exc}") from exc
if set(raw) != {"train", "validation", "test"}:
    raise ValueError(f"Cần đúng train/validation/test, nhận {list(raw)}")

def normalize_text(text):
    return " ".join(unicodedata.normalize("NFC", text).split())

def adapt_split(dataset, split):
    if not {"image", "text"}.issubset(dataset.column_names):
        raise ValueError("Schema IAM cần image và text.")
    dataset = dataset.cast_column("image", HFImage(decode=False))
    rows = []
    for index, row in enumerate(dataset):
        text = normalize_text(row["text"])
        if not text:
            raise ValueError(f"Transcription rỗng tại {split}:{index}; kiểm tra nguồn.")
        rows.append({"id": f"{split}:{index}", "image": row["image"], "text": text})
    return rows

train_records = adapt_split(raw["train"], "train")
val_records = adapt_split(raw["validation"], "validation")
test_records = adapt_split(raw["test"], "test")
SPLIT_FINGERPRINTS = {name: fingerprint([(r["id"], r["text"]) for r in records])
                      for name, records in (("train", train_records), ("validation", val_records), ("test", test_records))}
print("Số dòng theo split:", len(train_records), len(val_records), len(test_records))
print("Fingerprint:", SPLIT_FINGERPRINTS)
print("Ví dụ nhãn train:", train_records[0]["text"])
print("Giữ split có sẵn; HF không có writer_id nên chưa chứng minh được writer-disjoint.")


## 4. Alphabet, ảnh và Dataset

Fit alphabet chỉ train; blank=0, UNK=1. Giữ toàn dòng, resize theo tỷ lệ, padding trắng bên phải, mask/length không lấy phần PAD làm văn bản. Kiểm cả ký tự lặp trong điều kiện CTC; không zero-infinity để che mẫu bất khả thi.


In [ ]:
BLANK_ID, UNK_ID = 0, 1
characters = sorted({ch for record in train_records for ch in record["text"]})
char_vocab = {"<BLANK>": BLANK_ID, "<UNK>": UNK_ID}
char_vocab.update({ch: index + 2 for index, ch in enumerate(characters)})
id_to_char = {index: ch for ch, index in char_vocab.items()}
TOKENIZER_STATE = {"char_vocab": char_vocab, "blank_id": BLANK_ID, "unk_id": UNK_ID}
PREPROCESS_CONFIG = {"image_height": IMAGE_HEIGHT, "width_multiple": 4,
                     "normalization": "NFC, gộp khoảng trắng; giữ hoa/thường và dấu câu",
                     "image_normalization": "grayscale 0..1; white right padding required by variable-width CTC", "decoding": "CTC greedy"}

def restore_tokenizer(state):
    global char_vocab, id_to_char
    char_vocab = state["char_vocab"]
    id_to_char = {index: ch for ch, index in char_vocab.items()}

def image_tensor(value):
    image = open_rgb(value)
    width = max(32, 4 * math.ceil(image.width * IMAGE_HEIGHT / image.height / 4))
    # Giữ trọn dòng và tỷ lệ gần nguyên gốc, chỉ làm tròn width lên bội số 4.
    image = image.resize((width, IMAGE_HEIGHT), Image.Resampling.BILINEAR)
    tensor = TF.to_tensor(image.convert("L"))
    return tensor

class OCRDataset(Dataset):
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, index):
        record = self.records[index]
        target = torch.tensor([char_vocab.get(ch, UNK_ID) for ch in record["text"]], dtype=torch.long)
        return image_tensor(record["image"]), target, record

def collate_batch(items):
    widths = torch.tensor([image.shape[-1] for image, _, _ in items], dtype=torch.long)
    images = torch.ones(len(items), 1, IMAGE_HEIGHT, int(widths.max()))
    targets, lengths = [], []
    for i, (image, target, record) in enumerate(items):
        images[i, :, :, :image.shape[-1]] = image
        required = len(target) + int((target[1:] == target[:-1]).sum())
        if image.shape[-1] // 4 < required:
            raise ValueError(f"Dòng {record['id']} không đủ bước CTC. Tăng IMAGE_HEIGHT thủ công rồi Run All.")
        targets.append(target)
        lengths.append(len(target))
    return {"images": images, "widths": widths, "targets": torch.cat(targets),
            "target_lengths": torch.tensor(lengths), "ids": [r["id"] for _, _, r in items],
            "texts": [r["text"] for _, _, r in items]}

train_loader = make_loader(OCRDataset(train_records), train=True)
val_loader = make_loader(OCRDataset(val_records))
test_loader = make_loader(OCRDataset(test_records))
print("Số ký tự train (kể cả special):", len(char_vocab))
for name, records in (("validation", val_records), ("test", test_records)):
    unseen = sorted({ch for r in records for ch in r["text"] if ch not in char_vocab})
    print(f"Ký tự chưa gặp ở {name}: {unseen}; giữ gold nguyên văn, không bỏ ký tự khi chấm.")


## 5. Kiến trúc

CNN nhỏ tự tạo + RNN một chiều + CTC; toàn bộ học từ đầu, không có pretrained.


In [ ]:
class OCRModel(nn.Module):
    def __init__(self, cfg, initialize_pretrained=True):
        super().__init__()
        self.visual = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d((2, 1)))
        self.sequence = nn.RNN(128, cfg["hidden_dim"], nonlinearity="tanh", batch_first=True)
        self.head = nn.Linear(cfg["hidden_dim"], cfg["num_classes"])
    def forward(self, images, widths):
        x = self.visual(images).mean(dim=2).transpose(1, 2)
        lengths = widths // 4
        packed = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        hidden, _ = self.sequence(packed)
        hidden, _ = pad_packed_sequence(hidden, batch_first=True, total_length=x.size(1))
        return self.head(hidden).log_softmax(-1), lengths

def build_model(cfg, initialize_pretrained=True):
    return OCRModel(cfg, initialize_pretrained)


In [ ]:
MODEL_CONFIG = {"hidden_dim": HIDDEN_DIM, "dropout": DROPOUT, "num_classes": len(char_vocab)}


## 6. CER/WER và loss

CER/WER là tổng edit distance chia tổng số ký tự/từ gold toàn split, không trung bình theo batch. NFC, gộp khoảng trắng, giữ hoa/thường/dấu câu. CER/WER có thể lớn hơn 1; càng thấp càng tốt. Greedy CTC chỉ giải mã theo độ dài ảnh thật.


In [ ]:
def edit_distance(a, b):
    previous = list(range(len(b) + 1))
    for i, left in enumerate(a, 1):
        current = [i]
        for j, right in enumerate(b, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (left != right)))
        previous = current
    return previous[-1]

def ctc_decode(path):
    output, previous = [], None
    for index in path:
        index = int(index)
        if index != BLANK_ID and index != previous:
            output.append(id_to_char[index] if index != UNK_ID else "�")
        previous = index
    return normalize_text("".join(output))

ctc_loss = nn.CTCLoss(blank=BLANK_ID, reduction="mean", zero_infinity=False)
def compute_loss(model, batch):
    log_probs, lengths = model(batch["images"], batch["widths"])
    if int(lengths.max()) > log_probs.size(1):
        raise ValueError("Độ dài CTC không khớp feature map.")
    return ctc_loss(log_probs.transpose(0, 1), batch["targets"], lengths, batch["target_lengths"])

@torch.no_grad()
def evaluate(model, loader, return_predictions=False):
    model.eval()
    char_errors = word_errors = char_total = word_total = 0
    predictions = []
    for cpu_batch in loader:
        batch = to_device(cpu_batch)
        log_probs, lengths = model(batch["images"], batch["widths"])
        paths = log_probs.argmax(-1).cpu().tolist()
        for sid, gold, path, length in zip(batch["ids"], batch["texts"], paths, lengths.tolist()):
            prediction = ctc_decode(path[:length])
            char_errors += edit_distance(gold, prediction)
            word_errors += edit_distance(gold.split(), prediction.split())
            char_total += len(gold)
            word_total += len(gold.split())
            if return_predictions:
                predictions.append({"id": sid, "reference": gold, "prediction": prediction})
    if not char_total or not word_total:
        raise ValueError("Không có gold để tính CER/WER.")
    metrics = {"cer": char_errors / char_total, "wer": word_errors / word_total,
               "character_errors": char_errors, "reference_characters": char_total,
               "word_errors": word_errors, "reference_words": word_total}
    return (metrics, predictions) if return_predictions else metrics


## 7. Huấn luyện và checkpoint

AdamW, cosine LR, clipping, early stopping theo CER validation giảm. Lưu toàn bộ weights/alphabet cần cho inference; cấu hình/provenance trong notebook. Không chọn checkpoint theo train loss hoặc test.


In [ ]:


def make_optimizer(model):
    visual, remaining = [], []
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            (visual if name.startswith("backbone.") else remaining).append(parameter)
    groups = [{"params": remaining, "lr": LEARNING_RATE}]
    if visual:
        groups.append({"params": visual, "lr": BACKBONE_LEARNING_RATE})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)




In [ ]:
model = build_model(MODEL_CONFIG, initialize_pretrained=True).to(DEVICE)
print(f"Tham số học: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
optimizer = make_optimizer(model)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_score = math.inf if DIRECTION == "min" else -math.inf
stale_epochs = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total, seen = 0.0, 0
    for cpu_batch in tqdm(train_loader, desc="Huấn luyện", leave=False):
        batch = to_device(cpu_batch)
        optimizer.zero_grad(set_to_none=True)
        loss = compute_loss(model, batch)
        if not torch.isfinite(loss):
            raise FloatingPointError("Loss không hữu hạn; kiểm tra nhãn, mask và cấu hình cố định.")
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        count = len(batch["ids"])
        total += loss.item() * count
        seen += count
    if not seen:
        raise ValueError("Tập huấn luyện rỗng.")
    loss = total / seen
    metrics = evaluate(model, val_loader)
    score = float(metrics[PRIMARY_METRIC])
    if not math.isfinite(score):
        raise ValueError("Metric validation không hữu hạn.")
    scheduler.step()
    print(f"Epoch {epoch:02d}: loss={loss:.4f}; validation={metrics}")
    improved = score < best_score if DIRECTION == "min" else score > best_score
    if improved:
        best_score, stale_epochs = score, 0
        torch.save({
            "model_state": model.state_dict(),
            "epoch": epoch,
            "tokenizer": TOKENIZER_STATE,
        }, CHECKPOINT_PATH)
        print("Đã lưu checkpoint tốt nhất theo primary metric validation.")
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f"Dừng sớm sau {PATIENCE} epoch không cải thiện.")
            break


## 8. Tải lại và đánh giá test

Tạo mô hình mới không tải pretrained lại, phục hồi toàn bộ weights từ checkpoint rồi đánh giá test một lần. Chỉ ghi điểm khi thực sự chạy.


In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
assert checkpoint["tokenizer"] == TOKENIZER_STATE
restore_tokenizer(checkpoint["tokenizer"])
# Không tải backbone lần nữa: state_dict chứa đầy đủ trọng số visual và head.
# Cấu hình kiến trúc ở notebook; kích thước vocab được lấy sau khi restore.
inference_config = {"hidden_dim": HIDDEN_DIM, "dropout": DROPOUT, "num_classes": len(char_vocab)}
best_model = build_model(inference_config, initialize_pretrained=False).to(DEVICE)
best_model.load_state_dict(checkpoint["model_state"])
best_model.eval()
validation_metrics = evaluate(best_model, val_loader)
test_metrics, predictions = evaluate(best_model, test_loader, return_predictions=True)
results = {"epoch": checkpoint["epoch"], "source": SOURCE, "split_fingerprints": SPLIT_FINGERPRINTS,
           "validation": validation_metrics, "held_out_test": test_metrics}
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "predictions.json").write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Đã nạp epoch {checkpoint['epoch']}.")
print("Validation:", validation_metrics)
print("Held-out:", test_metrics)
print("Ví dụ dự đoán:", predictions[0] if predictions else "Không có mẫu.")


## 9. Đọc dòng mới

Truyền ảnh PIL hoặc đường dẫn; không dùng transcription để tính độ rộng ảnh khi inference.


In [ ]:
@torch.no_grad()
def recognize_line(image):
    tensor = image_tensor(image)
    width = torch.tensor([tensor.shape[-1]])
    best_model.eval()
    log_probs, lengths = best_model(tensor.unsqueeze(0).to(DEVICE), width)
    return ctc_decode(log_probs[0, :int(lengths[0])].argmax(-1).tolist())

print("Đọc dòng đầu test:", recognize_line(test_records[0]["image"]))
